## Clone private GitHub repo (optional)

Repo mặc định là `Dle28/nlp-finance-query-`. Nếu repo private, đặt secret `GIT_TOKEN` trong Kaggle trước khi chạy. Token không được in hoặc lưu vào remote URL.

In [ ]:
from pathlib import Path
import os
import subprocess
import base64

REPO_SLUG = 'Dle28/nlp-finance-query-'
GIT_TOKEN_SECRET_NAME = 'GIT_TOKEN'
REPO_DIR = Path('/kaggle/working/AI_guru')
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret(GIT_TOKEN_SECRET_NAME)
except Exception:
    pass

git_env = os.environ.copy()
git_env['GIT_TERMINAL_PROMPT'] = '0'
basic = None
if token:
    # Supply the secret only to this subprocess; it is never written to origin.
    basic = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    git_env.update({
        'GIT_CONFIG_COUNT': '1',
        'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
        'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {basic}',
    })

try:
    if REPO_DIR.exists():
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', 'main'], env=git_env, check=True)
        print('Repo updated:', REPO_DIR)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', f'https://github.com/{REPO_SLUG}.git', str(REPO_DIR)], env=git_env, check=True)
finally:
    token = None
    basic = None
    git_env = None
print('Repo ready:', REPO_DIR)

# ViFinQA P2 — Kaggle GPU benchmark

Notebook này chỉ đo throughput embedding/training và peak VRAM trên GPU Kaggle. Nó không rebuild raw corpus, V2/V3 tables, lexical index hoặc dense index.

Trước khi chạy, attach private Kaggle Dataset `dungle2810/vifinqa-baseline-artifacts` ở panel **Input**. Notebook sẽ dùng `kaggle/bootstrap.py` để khôi phục các artifact vào repo. Nếu chưa attach dataset, cell chuẩn bị dữ liệu sẽ dừng với hướng dẫn cụ thể.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_DIR = Path('/kaggle/working/AI_guru')
if not REPO_DIR.is_dir():
    raise FileNotFoundError(f'Không tìm thấy repo: {REPO_DIR}. Hãy clone/upload repo trước.')

# Kaggle hiện có thể cài wheel CUDA 13 mặc định, vốn không chứa Pascal sm_60.
# Wheel CUDA 12.6 legacy của PyTorch vẫn hỗ trợ P100/Pascal. Cài trước khi import torch.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    'torch==2.12.1', 'torchvision==0.27.1',
    '--index-url', 'https://download.pytorch.org/whl/cu126',
], check=True)

# Khóa stack text-only: sentence-transformers 5.x tự import torchcodec/FFmpeg
# dù benchmark này không xử lý audio/video. Bản 3.4.1 vẫn hỗ trợ API fit hiện tại.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    'sentence-transformers==3.4.1', 'transformers==4.48.3',
], check=True)

import torch

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Kaggle chưa bật Accelerator GPU.')
gpu_name = torch.cuda.get_device_name(0)
gpu_capability = torch.cuda.get_device_capability(0)
gpu_arch = f'sm_{gpu_capability[0]}{gpu_capability[1]}'
print('GPU:', gpu_name)
print('Compute capability:', gpu_capability, gpu_arch)
print('CUDA:', torch.version.cuda)
if gpu_arch not in torch.cuda.get_arch_list():
    raise RuntimeError(
        f'PyTorch hiện tại không có kernel {gpu_arch} cho {gpu_name}. '
        'Hãy Restart Session rồi chạy lại từ đầu để nạp wheel CUDA 12.6 vừa cài.'
    )
!nvidia-smi

In [ ]:
REPO_ASSETS = REPO_DIR / 'artifacts' / 'table_assets.jsonl'
if not REPO_ASSETS.is_file():
    bootstrap_script = REPO_DIR / 'kaggle' / 'bootstrap.py'
    if not bootstrap_script.is_file():
        raise FileNotFoundError(f'Không tìm thấy bootstrap script: {bootstrap_script}')
    print('Chưa có baseline artifacts trong repo; đang khôi phục từ Kaggle Input...')
    bootstrap = subprocess.run([sys.executable, str(bootstrap_script)], cwd=REPO_DIR)
    if bootstrap.returncode:
        raise FileNotFoundError(
            'Chưa attach dataset baseline. Vào panel Input → Add Input và attach '
            '`dungle2810/vifinqa-baseline-artifacts`, sau đó Restart Session và Run All.'
        )

# --no-deps tránh pip thay stack PyTorch/sentence-transformers vừa pin.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR), '--no-deps'], check=True)

dependency_check = subprocess.run(
    [
        sys.executable, '-c',
        'import torch, transformers, sentence_transformers; '
        'print("Dependency check:", torch.__version__, transformers.__version__, sentence_transformers.__version__)',
    ],
    capture_output=True, text=True,
)
print(dependency_check.stdout.strip())
if dependency_check.returncode:
    print(dependency_check.stderr)
    raise RuntimeError('Dependency preflight failed; Restart Session rồi chạy lại từ đầu.')

search_roots = [REPO_DIR / 'artifacts', Path('/kaggle/input'), Path('/kaggle/working')]
asset_candidates = sorted(
    [p for root in search_roots if root.exists() for p in root.rglob('table_assets.jsonl')],
    key=lambda p: p.stat().st_size,
    reverse=True,
)
if not asset_candidates:
    raise FileNotFoundError(
        'Không tìm thấy table_assets.jsonl. Attach `dungle2810/vifinqa-baseline-artifacts` '
        'và chạy lại cell này; notebook không tự rebuild corpus.'
    )
ASSETS = asset_candidates[0]
print('Using assets:', ASSETS)
print('Candidate count:', len(asset_candidates))

In [ ]:
import json
import time

RESULT_DIR = Path('/kaggle/working/vifinqa_gpu_benchmark_results')
RESULT_DIR.mkdir(parents=True, exist_ok=True)

def parse_benchmark_result(stdout):
    """Return the final benchmark JSON even if a library emitted log lines first."""
    decoder = json.JSONDecoder()
    candidates = []
    for start, character in enumerate(stdout):
        if character != '{':
            continue
        try:
            payload, _ = decoder.raw_decode(stdout[start:])
        except json.JSONDecodeError:
            continue
        if isinstance(payload, dict) and {'model', 'device'} <= payload.keys():
            candidates.append(payload)
    if candidates:
        return candidates[-1]
    raise ValueError(f'Benchmark did not emit a JSON result. stdout tail: {stdout[-2000:]}')

def run_benchmark(name, *, encode_only=False):
    command = [
        sys.executable, str(REPO_DIR / 'scripts' / 'benchmark_runtime.py'),
        '--assets', str(ASSETS), '--device', 'cuda:0', '--gpu-id', '0',
        '--sample-size', '256', '--batch-size', '16',
    ]
    if encode_only:
        command.append('--encode-only')
    else:
        command += ['--train-batch-size', '2', '--train-steps', '5', '--train-pairs', '1000', '--epochs', '3', '--gradient-checkpointing']
    environment = os.environ.copy()
    environment['CUDA_VISIBLE_DEVICES'] = '0'
    environment['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    environment['WANDB_DISABLED'] = 'true'
    environment['WANDB_MODE'] = 'disabled'
    started = time.perf_counter()
    completed = subprocess.run(command, cwd=REPO_DIR, env=environment, capture_output=True, text=True)
    elapsed = time.perf_counter() - started
    print(completed.stdout)
    if completed.returncode:
        print(completed.stderr)
        raise RuntimeError(f'Benchmark {name} failed with exit code {completed.returncode}')
    result = parse_benchmark_result(completed.stdout)
    result['wall_seconds_including_startup'] = elapsed
    result['benchmark_name'] = name
    output = RESULT_DIR / f'{name}.json'
    output.write_text(json.dumps(result, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    print('Saved:', output)
    return result

## 1. Encoding throughput

Đây là phép đo gần với dense indexing nhưng không tạo index mới.

In [ ]:
encode_result = run_benchmark('encode_only', encode_only=True)
encode_result

## 2. Synthetic training throughput

Cặp query/passage là synthetic để đo tốc độ; output model không được dùng làm nhãn hay artifact production.

In [ ]:
training_result = run_benchmark('synthetic_training', encode_only=False)
training_result

In [ ]:
summary = {
    'gpu': torch.cuda.get_device_name(0),
    'encode_tables_per_second': encode_result.get('tables_per_second'),
    'estimated_dense_index_hours': encode_result.get('estimated_full_dense_index_hours'),
    'peak_encode_vram_mb': encode_result.get('peak_gpu_memory_allocated_mb'),
    'seconds_per_training_step': training_result.get('seconds_per_training_step'),
    'estimated_training_hours': training_result.get('estimated_training_hours'),
    'peak_training_vram_mb': training_result.get('peak_gpu_memory_allocated_mb'),
}
(RESULT_DIR / 'summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(json.dumps(summary, ensure_ascii=False, indent=2))

# 3. Qwen 2.5 14B — evidence-bounded staged review

Attach the **full review bundle** Kaggle Dataset before running this section (it must contain `review_items.jsonl`, `tables_structured_v2.jsonl`, and `tables_evidence_context_v3.jsonl`). Turn **Internet on** so Kaggle can download the public Qwen model. This section rebuilds navigation sidecars only; it never edits raw reports or creates a submission answer.

In [ ]:
import shutil

REVIEW_REQUIRED = {'review_items.jsonl', 'tables.jsonl', 'tables_structured_v2.jsonl', 'tables_evidence_context_v3.jsonl'}
WORKING_ROOT = Path('/kaggle/working')
INPUT_ROOT = Path('/kaggle/input')
DEFAULT_REVIEW_BUNDLE = WORKING_ROOT / 'vifinqa_review_bundle'

def is_complete_review_bundle(directory):
    return directory.is_dir() and all((directory / name).is_file() for name in REVIEW_REQUIRED)

# Prefer a bundle already materialized in writable storage, then a complete Kaggle Input.
working_sources = [DEFAULT_REVIEW_BUNDLE] if is_complete_review_bundle(DEFAULT_REVIEW_BUNDLE) else []
input_sources = sorted({
    path.parent for path in INPUT_ROOT.rglob('review_items.jsonl')
    if is_complete_review_bundle(path.parent)
})
review_sources = [*working_sources, *[source for source in input_sources if source not in working_sources]]
if not review_sources:
    input_datasets = sorted(path.name for path in INPUT_ROOT.iterdir() if path.is_dir())
    raise FileNotFoundError(
        'Không tìm thấy full ViFinQA review bundle. Dataset `vifinqa-baseline-artifacts` chỉ có index/asset '
        'phục vụ benchmark và không có review_items.jsonl cùng V2/V3 provenance. Trong Input → Add Input, '
        'hãy attach dataset chứa: ' + ', '.join(sorted(REVIEW_REQUIRED)) + '. '
        f'Input hiện có: {input_datasets or ["(none)"]}'
    )
if len(review_sources) > 1:
    print('Multiple complete review bundles found; using:', review_sources[0])
REVIEW_SOURCE = review_sources[0]
if REVIEW_SOURCE == DEFAULT_REVIEW_BUNDLE:
    REVIEW_BUNDLE = REVIEW_SOURCE
    print('Using existing writable review bundle:', REVIEW_BUNDLE)
else:
    REVIEW_BUNDLE = DEFAULT_REVIEW_BUNDLE
    if REVIEW_BUNDLE.exists():
        raise RuntimeError(f'Existing incomplete working bundle: {REVIEW_BUNDLE}. Restart Session or remove only that directory.')
    print('Copying read-only Kaggle bundle to working storage:', REVIEW_SOURCE)
    shutil.copytree(REVIEW_SOURCE, REVIEW_BUNDLE)
print('Review bundle:', REVIEW_BUNDLE)

# Keep the P100-compatible torch stack installed in the earlier cells.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'accelerate>=1.0', 'bitsandbytes>=0.45'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR), '--no-deps'], check=True)
if torch.cuda.get_device_properties(0).total_memory < 14 * 1024**3:
    raise RuntimeError('Qwen2.5-14B 4-bit review requires at least 14 GiB GPU memory.')
print('Qwen GPU preflight:', torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), 'GiB')

In [ ]:
QWEN_RESULTS = Path('/kaggle/working/vifinqa_qwen_review_results')
QWEN_RESULTS.mkdir(parents=True, exist_ok=True)
ROUTES = QWEN_RESULTS / 'staged_retrieval_routes_v1.jsonl'
QWEN_QUESTION_IDS = ('368', '369')

# Source-preserving preprocessing: metadata, table type, variables, fiscal year end.
subprocess.run([sys.executable, 'scripts/build_report_normalization.py', '--bundle-dir', str(REVIEW_BUNDLE), '--force'], cwd=REPO_DIR, check=True)
subprocess.run([sys.executable, 'scripts/build_staged_retrieval_routes.py', '--questions', str(REVIEW_BUNDLE / 'review_items.jsonl'), '--output', str(ROUTES)], cwd=REPO_DIR, check=True)

# No model download: inspect the exact source-cell packet before inference.
DRY_RUN = QWEN_RESULTS / 'qwen14_staged_review_dry_run_v1.jsonl'
subprocess.run([
    sys.executable, 'scripts/run_qwen_staged_review.py',
    '--bundle-dir', str(REVIEW_BUNDLE), '--routes', str(ROUTES), '--output', str(DRY_RUN),
    *[flag for question_id in QWEN_QUESTION_IDS for flag in ('--question-id', question_id)], '--dry-run',
], cwd=REPO_DIR, check=True)
print((DRY_RUN.with_suffix('.packets.jsonl')).read_text(encoding='utf-8')[:3000])

In [ ]:
# Qwen selects only literal cells from the packet. Deterministic code computes
# Q368: Quick Ratio -> median -> Net Profit Margin average.
# Q369: Quick Ratio -> Gross Profit Margin change -> Interest Coverage.
# The output remains machine_provisional and requires independent replay/critic.
RUN_QWEN14 = True
QWEN_OUTPUT = QWEN_RESULTS / 'qwen14_staged_review_v1.jsonl'
if RUN_QWEN14:
    review_env = os.environ.copy()
    review_env.update({'CUDA_VISIBLE_DEVICES': '0', 'WANDB_DISABLED': 'true', 'WANDB_MODE': 'disabled'})
    subprocess.run([
        sys.executable, 'scripts/run_qwen_staged_review.py',
        '--bundle-dir', str(REVIEW_BUNDLE), '--routes', str(ROUTES), '--output', str(QWEN_OUTPUT),
        '--model', 'Qwen/Qwen2.5-14B-Instruct',
        *[flag for question_id in QWEN_QUESTION_IDS for flag in ('--question-id', question_id)],
        '--max-tables-per-binding', '2', '--max-new-tokens', '1536',
    ], cwd=REPO_DIR, env=review_env, check=True)
    print(QWEN_OUTPUT.read_text(encoding='utf-8'))
else:
    print('Qwen inference disabled; dry-run packet is available at', DRY_RUN)

## 4. Independent source audit

This reopens Qwen-selected V2/V3 cells and independently recomputes both stages from the bounded source packet. A dry run or test fixture is intentionally not promotable. Even a `machine_calibrated` audit output is not submission/training eligible.

In [ ]:
QWEN_AUDIT = QWEN_RESULTS / 'qwen14_staged_review_audit_v1.jsonl'
if RUN_QWEN14:
    subprocess.run([
        sys.executable, 'scripts/audit_qwen_staged_review.py',
        '--bundle-dir', str(REVIEW_BUNDLE), '--routes', str(ROUTES),
        '--qwen-output', str(QWEN_OUTPUT), '--output', str(QWEN_AUDIT),
        *[flag for question_id in QWEN_QUESTION_IDS for flag in ('--question-id', question_id)],
    ], cwd=REPO_DIR, check=True)
    print(QWEN_AUDIT.read_text(encoding='utf-8'))
else:
    print('Independent audit waits for a completed non-dry-run Qwen output.')